# Firewatch SFT Training - Colab

Thin launcher for the real Hub-backed SFT pipeline. Durable artifacts are written to Hugging Face Hub model repos, not to the Colab VM.

## 1. Runtime Setup

Select a GPU runtime, add `HF_TOKEN` in Colab Secrets, then run each cell in order.

In [ ]:
!nvidia-smi
!pip install -q virtualenv

In [ ]:
from google.colab import userdata
import os

token = userdata.get('HF_TOKEN')
if not token:
    raise RuntimeError('Set HF_TOKEN in Colab Secrets before running training.')
os.environ['HF_TOKEN'] = token
os.environ.setdefault('HF_HUB_DISABLE_TELEMETRY', '1')

## 2. Repository Setup

Clone the project if it is not already mounted. Update `REPO_URL` for your working branch or fork.

In [ ]:
import shutil
from pathlib import Path

REPO_URL = 'https://github.com/10doshi12/firewatch_agent.git'
REPO_BRANCH = 'main'
repo_dir = Path('/content/firewatch_agent')
if repo_dir.exists():
    # Always pull on a re-run — without this, a stale clone silently runs out-of-date code.
    !cd {repo_dir} && git fetch origin {REPO_BRANCH} && git reset --hard origin/{REPO_BRANCH}
else:
    !git clone --branch {REPO_BRANCH} {REPO_URL} {repo_dir}
%cd /content/firewatch_agent
venv_path = repo_dir / '.venv'
if venv_path.exists():
    shutil.rmtree(venv_path)
!python -m virtualenv .venv
!.venv/bin/python -m pip install --upgrade pip setuptools wheel
# Do not run `uv sync` here: the lockfile currently resolves a CUDA-13 torch stack,
# which conflicts with Unsloth's CUDA-12 stack on Colab T4.
!.venv/bin/python -m pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!.venv/bin/python -m pip install --no-deps torchvision==0.25.0
# Install the project itself without re-resolving the torch/transformers stack, then add
# the small set of project deps that Unsloth does not bring along.
!.venv/bin/python -m pip install --no-deps -e .
!.venv/bin/python -m pip install jsonschema python-dotenv websockets matplotlib torch-geometric hf-transfer
# Sanity checks — fail fast if the venv still has a mixed CUDA/NCCL stack.
!.venv/bin/python -c "import torch, torchvision; print('[notebook] torch', torch.__version__, 'torchvision', torchvision.__version__, 'cuda', torch.cuda.is_available(), 'device', torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)"
!.venv/bin/python -c "import unsloth; print('[notebook] unsloth importable:', unsloth.__version__)"

## 3. Preflight Then Train

Preflight must pass before the base LLM is loaded. This notebook builds a clean pip virtualenv, installs Unsloth plus torchvision first, then installs the project without re-resolving Torch.

In [ ]:
!.venv/bin/python -m sft.preflight --config config.yaml
!.venv/bin/python -m sft.train --config config.yaml

Expected outputs:
- `<namespace>/firewatch-gnn/gnn/batch_NNN.pt`
- `<namespace>/firewatch-gnn/gnn/normalization.json`
- `<namespace>/firewatch-agent-sft/batch_NNN/`